# 🌱 Plataforma de Agricultura Inteligente Macollo
## Análisis Exploratorio de Datos y Transformación

**Descripción del Proyecto:**
Este notebook realiza un análisis completo de los datos del invernadero Macollo, incluyendo:
- Ingesta y limpieza de datos (ETL)
- Análisis Exploratorio de Datos (EDA) con visualizaciones
- Transformación de variables categóricas a numéricas
- Dashboard interactivo para consultas por fecha

**Autor:** Equipo de Data Science - Agricultura Inteligente  
**Fecha:** 2026

##  Tabla de Contenidos
1. [Importación de Librerías](#importacion)
2. [Proceso ETL](#etl)
3. [Análisis Exploratorio de Datos (EDA)](#eda)
4. [Transformación de Datos](#transformacion)
5. [Dashboard por Fecha](#dashboard)

---
## 1. Importación de Librerías <a id='importacion'></a>

En esta sección importamos todas las librerías necesarias para el análisis de datos y visualizaciones.

In [ ]:
%matplotlib inline

# Librerías para manipulación de datos
import pandas as pd
import numpy as np

# Librerías para visualización
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Librerías para transformación de datos
from sklearn.preprocessing import OneHotEncoder

# Configuración de visualizaciones
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Librerías importadas exitosamente")

---
## 2. Proceso ETL (Extract, Transform, Load) <a id='etl'></a>

### 2.1 Extracción de Datos
Cargamos el dataset desde el archivo CSV y realizamos una primera inspección.

In [ ]:
# Cargar el dataset
df = pd.read_csv('Data_Mocollo1.csv', sep=';')

# Mostrar información básica del dataset
print("=" * 60)
print("📊 INFORMACIÓN BÁSICA DEL DATASET")
print("=" * 60)
print(f"\n📏 Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\n📋 Columnas disponibles:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

print(f"\n Primeras 5 filas del dataset:")
df.head()

### 2.2 Transformación y Limpieza de Datos

Realizamos las siguientes transformaciones:
- Conversión de fechas
- Manejo de valores numéricos con formato europeo (coma como decimal)
- Detección y tratamiento de valores atípicos
- Manejo de valores faltantes
- Normalización de escala de la humedad relativa (0-1)

In [ ]:
# Función para convertir formato europeo a decimal estándar
def convertir_a_numero(valor):
    """
    Convierte valores con formato europeo (coma decimal) a float
    Maneja casos especiales como fechas en formato numérico Excel
    """
    if pd.isna(valor):
        return np.nan
    
    try:
        # Si es string con coma, convertir a punto
        if isinstance(valor, str):
            valor = valor.replace(',', '.')
            # Verificar si es una fecha de Excel (número grande)
            num = float(valor)
            if num > 10000:  # Probable fecha de Excel
                return np.nan  # Marcar como NaN para manejar después
            return num
        return float(valor)
    except:
        return np.nan

# Crear copia del dataframe para trabajar
df_clean = df.copy()

# Convertir columnas numéricas
columnas_numericas = ['T_Ext_C', 'T_Int_C', 'Humedad_Pct', 'T_Planta_C', 'pH', 'Altura_cm']

print("🔄 Convirtiendo columnas numéricas...")
for col in columnas_numericas:
    df_clean[col] = df_clean[col].apply(convertir_a_numero)

# Convertir fecha
df_clean['Fecha'] = pd.to_datetime(df_clean['Fecha'], format='%d/%m/%Y', errors='coerce')

# Normalizar humedad relativa: algunos lotes se registraron como porcentaje (0-100)
mask_humedad = df_clean['Humedad_Pct'] > 1
humedad_corregida = int(mask_humedad.sum())
df_clean.loc[mask_humedad, 'Humedad_Pct'] = df_clean.loc[mask_humedad, 'Humedad_Pct'] / 100


print(f"\n✅ Conversión completada")
print(f"🔧 Registros de humedad normalizados: {humedad_corregida}")
print(f"📊 Valores nulos por columna:")
print(df_clean[columnas_numericas].isnull().sum())

In [ ]:
# Estadísticas descriptivas básicas
print("=" * 60)
print("📈 ESTADÍSTICAS DESCRIPTIVAS")
print("=" * 60)
df_clean.describe()

---
## 3. Análisis Exploratorio de Datos (EDA) <a id='eda'></a>

Realizamos un análisis visual completo de los datos para entender:
- Distribuciones de variables
- Relaciones entre variables
- Valores atípicos
- Patrones temporales

### 3.1 Visualización de Variables Categóricas

In [ ]:
# Crear figura con subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📊 Distribución de Variables Categóricas', fontsize=16, fontweight='bold')

# 1. Distribución por Responsable (Gráfico de barras)
ax1 = axes[0, 0]
responsable_counts = df_clean['Responsable'].value_counts()
colors1 = plt.cm.Set3(np.linspace(0, 1, len(responsable_counts)))
ax1.bar(responsable_counts.index, responsable_counts.values, color=colors1, edgecolor='black')
ax1.set_title('Mediciones por Responsable', fontweight='bold')
ax1.set_xlabel('Responsable')
ax1.set_ylabel('Número de Mediciones')
ax1.tick_params(axis='x', rotation=45)

# Añadir etiquetas de valor en las barras
for i, v in enumerate(responsable_counts.values):
    ax1.text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

# 2. Distribución por Fila (Gráfico de pastel)
ax2 = axes[0, 1]
fila_counts = df_clean['Fila'].value_counts()
colors2 = plt.cm.Pastel1(np.linspace(0, 1, len(fila_counts)))
ax2.pie(fila_counts.values, labels=fila_counts.index, autopct='%1.1f%%', 
        colors=colors2, startangle=90, explode=[0.05]*len(fila_counts))
ax2.set_title('Distribución por Fila/Treatment', fontweight='bold')

# 3. Distribución por Planta (Gráfico de barras horizontales)
ax3 = axes[1, 0]
planta_counts = df_clean['Planta'].value_counts().sort_index()
colors3 = plt.cm.Set2(np.linspace(0, 1, len(planta_counts)))
ax3.barh(planta_counts.index, planta_counts.values, color=colors3, edgecolor='black')
ax3.set_title('Mediciones por Planta', fontweight='bold')
ax3.set_xlabel('Número de Mediciones')
ax3.set_ylabel('Planta')

# Añadir etiquetas de valor
for i, v in enumerate(planta_counts.values):
    ax3.text(v, i, f' {v}', va='center', fontweight='bold')

# 4. ID_Planta únicos (Gráfico de líneas)
ax4 = axes[1, 1]
id_counts = df_clean['ID_Planta'].value_counts()
ax4.plot(range(len(id_counts)), id_counts.values, 'o-', linewidth=2, markersize=4)
ax4.set_title('Distribución de ID_Planta', fontweight='bold')
ax4.set_xlabel('Índice de ID_Planta')
ax4.set_ylabel('Frecuencia')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n OBSERVACIONES:")
print(f"• Total de responsables: {df_clean['Responsable'].nunique()}")
print(f"• Total de filas/treatments: {df_clean['Fila'].nunique()}")
print(f"• Total de plantas: {df_clean['Planta'].nunique()}")
print(f"• Total de IDs únicos: {df_clean['ID_Planta'].nunique()}")

### 3.2 Histogramas de Variables Numéricas

Analizamos la distribución de las variables continuas para entender su comportamiento.

In [ ]:
# Crear figura para histogramas
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('📊 Histogramas de Variables Numéricas', fontsize=16, fontweight='bold')

# Lista de variables numéricas a graficar
variables = ['T_Ext_C', 'T_Int_C', 'Humedad_Pct', 'T_Planta_C', 'pH', 'Altura_cm']
titulos = ['Temperatura Externa (°C)', 'Temperatura Interna (°C)', 'Humedad (%)',
          'Temperatura Planta (°C)', 'pH', 'Altura (cm)']
colores = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F']

for idx, (var, title, color) in enumerate(zip(variables, titulos, colores)):
    ax = axes[idx // 3, idx % 3]
    
    # Crear histograma
    datos = df_clean[var].dropna()
    ax.hist(datos, bins=20, color=color, edgecolor='black', alpha=0.7)
    ax.axvline(datos.mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {datos.mean():.2f}')
    ax.axvline(datos.median(), color='blue', linestyle='--', linewidth=2, label=f'Mediana: {datos.median():.2f}')
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Imprimir estadísticas
print("\n" + "=" * 80)
print("📈 ESTADÍSTICAS DE VARIABLES NUMÉRICAS")
print("=" * 80)
for var, title in zip(variables, titulos):
    datos = df_clean[var].dropna()
    print(f"\n{title}:")
    print(f"  Media: {datos.mean():.2f} | Mediana: {datos.median():.2f}")
    print(f"  Desv. Estándar: {datos.std():.2f}")
    print(f"  Mín: {datos.min():.2f} | Máx: {datos.max():.2f}")

### 3.3 Gráficos de Caja y Bigotes (Boxplots)

Identificamos valores atípicos y comparamos distribuciones entre diferentes categorías.

In [ ]:
# Crear figura para boxplots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('📦 Distribución por Fila/Treatment (Boxplots)', fontsize=16, fontweight='bold')

variables = ['T_Ext_C', 'T_Int_C', 'Humedad_Pct', 'T_Planta_C', 'pH', 'Altura_cm']
titulos = ['Temperatura Externa (°C)', 'Temperatura Interna (°C)', 'Humedad (%)',
          'Temperatura Planta (°C)', 'pH', 'Altura (cm)']

for idx, (var, title) in enumerate(zip(variables, titulos)):
    ax = axes[idx // 3, idx % 3]
    
    # Crear boxplot por fila
    sns.boxplot(data=df_clean, x='Fila', y=var, ax=ax, palette='Set3')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Fila/Treatment')
    ax.set_ylabel('Valor')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Detectar outliers
print("\n🔍 DETECCIÓN DE VALORES ATÍPICOS (Outliers)")
print("=" * 60)
for var in variables:
    Q1 = df_clean[var].quantile(0.25)
    Q3 = df_clean[var].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df_clean[(df_clean[var] < Q1 - 1.5*IQR) | (df_clean[var] > Q3 + 1.5*IQR)]
    print(f"{var}: {len(outliers)} valores atípicos detectados")

### 3.4 Gráficos de Densidad (KDE)

Visualizamos la distribución de probabilidad de las variables continuas.

In [ ]:
# Crear figura para gráficos de densidad
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('📈 Gráficos de Densidad (KDE) por Treatment', fontsize=16, fontweight='bold')

variables = ['T_Ext_C', 'T_Int_C', 'Humedad_Pct', 'T_Planta_C', 'pH', 'Altura_cm']
titulos = ['Temperatura Externa (°C)', 'Temperatura Interna (°C)', 'Humedad (%)',
          'Temperatura Planta (°C)', 'pH', 'Altura (cm)']

for idx, (var, title) in enumerate(zip(variables, titulos)):
    ax = axes[idx // 3, idx % 3]
    
    # Crear KDE por fila
    for fila in df_clean['Fila'].unique():
        datos = df_clean[df_clean['Fila'] == fila][var].dropna()
        sns.kdeplot(data=datos, label=fila, ax=ax, fill=True, alpha=0.3)
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Densidad')
    ax.legend(title='Fila')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.5 Análisis Temporal

Analizamos la evolución de las mediciones a lo largo del tiempo.

In [ ]:
# Agrupar por fecha y calcular promedios
df_temporal = df_clean.groupby('Fecha').agg({
    'T_Ext_C': 'mean',
    'T_Int_C': 'mean',
    'Humedad_Pct': 'mean',
    'T_Planta_C': 'mean',
    'Altura_cm': 'mean'
}).reset_index()

# Crear figura
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('📅 Evolución Temporal de Variables', fontsize=16, fontweight='bold')

variables = ['T_Ext_C', 'T_Int_C', 'Humedad_Pct', 'T_Planta_C', 'Altura_cm']
titulos = ['Temperatura Externa (°C)', 'Temperatura Interna (°C)', 'Humedad (%)',
          'Temperatura Planta (°C)', 'Altura Promedio (cm)']
colores = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']

for idx, (var, title, color) in enumerate(zip(variables, titulos, colores)):
    ax = axes[idx // 2, idx % 2]
    
    # Línea de tendencia
    ax.plot(df_temporal['Fecha'], df_temporal[var], 'o-', 
           color=color, linewidth=2, markersize=6, label='Promedio')
    
    # Línea de tendencia suave
    ax.plot(df_temporal['Fecha'], df_temporal[var].rolling(window=3).mean(), 
           '--', color=color, alpha=0.5, linewidth=2, label='Media móvil (3 días)')
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Valor')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

# Espacio vacío
axes[2, 1].axis('off')

plt.tight_layout()
plt.show()

print(f"\n📅 Período de mediciones: {df_clean['Fecha'].min().strftime('%d/%m/%Y')} - {df_clean['Fecha'].max().strftime('%d/%m/%Y')}")
print(f"📊 Total de días con mediciones: {df_clean['Fecha'].nunique()}")

### 3.6 Matriz de Correlación

Analizamos las relaciones entre las diferentes variables numéricas.

In [ ]:
# Calcular matriz de correlación
variables_corr = ['T_Ext_C', 'T_Int_C', 'Humedad_Pct', 'T_Planta_C', 'pH', 'Altura_cm']
matriz_corr = df_clean[variables_corr].corr()

# Subconjunto alineado (sin nulos) para dispersión y línea de tendencia
df_scatter = df_clean[['T_Ext_C', 'T_Int_C', 'Fila']].dropna()

# Crear figura
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🔗 Matriz de Correlación entre Variables', fontsize=16, fontweight='bold')

# Heatmap
ax1 = axes[0]
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', center=0,
           fmt='.2f', square=True, linewidths=0.5, ax=ax1,
           xticklabels=['T_Ext', 'T_Int', 'Humedad', 'T_Planta', 'pH', 'Altura'],
           yticklabels=['T_Ext', 'T_Int', 'Humedad', 'T_Planta', 'pH', 'Altura'])
ax1.set_title('Coeficiente de Correlación de Pearson', fontweight='bold')

# Gráfico de dispersión entre variables más correlacionadas
ax2 = axes[1]
sns.scatterplot(data=df_scatter, x='T_Ext_C', y='T_Int_C',
               hue='Fila', alpha=0.6, s=100, ax=ax2)
ax2.set_title('Temperatura Externa vs Interna', fontweight='bold')
ax2.set_xlabel('Temperatura Externa (°C)')
ax2.set_ylabel('Temperatura Interna (°C)')
ax2.grid(True, alpha=0.3)

# Añadir línea de tendencia
z = np.polyfit(df_scatter['T_Ext_C'], df_scatter['T_Int_C'], 1)
p = np.poly1d(z)
ax2.plot(df_scatter['T_Ext_C'], p(df_scatter['T_Ext_C']),
        "r--", alpha=0.8, linewidth=2, label=f'Tendencia (r={matriz_corr.loc["T_Ext_C", "T_Int_C"]:.2f})')
ax2.legend()

plt.tight_layout()
plt.show()

print("\n🔍 CORRELACIONES MÁS IMPORTANTES:")
print("=" * 60)
# Ordenar correlaciones con Altura
corr_altura = matriz_corr['Altura_cm'].abs().sort_values(ascending=False)
for var, valor in corr_altura.items():
    if var != 'Altura_cm':
        print(f"{var} vs Altura: {valor:.3f}")


---
## 4. Transformación de Datos - One Hot Encoding <a id='transformacion'></a>

Convertimos las variables categóricas a numéricas utilizando la técnica de One Hot Encoding.
Esto es necesario para algoritmos de machine learning que requieren entrada numérica.

In [ ]:
# Crear copia para transformación
df_transformado = df_clean.copy()

print("=" * 70)
print("🔄 TRANSFORMACIÓN DE VARIABLES CATEGÓRICAS - ONE HOT ENCODING")
print("=" * 70)

# Identificar variables categóricas
variables_categoricas = ['Responsable', 'Fila', 'Planta', 'ID_Planta']
print(f"\n📋 Variables categóricas a transformar: {variables_categoricas}")

# Método 1: One Hot Encoding con pandas get_dummies
print("\n🔧 Aplicando One Hot Encoding...")
df_encoded = pd.get_dummies(df_transformado, 
                           columns=variables_categoricas,
                           prefix=variables_categoricas,
                           drop_first=False)

print(f"\n✅ Transformación completada exitosamente")
print(f"📊 Dimensiones originales: {df_transformado.shape}")
print(f" Dimensiones después del encoding: {df_encoded.shape}")
print(f"➕ Columnas añadidas: {df_encoded.shape[1] - df_transformado.shape[1]}")

# Mostrar las nuevas columnas creadas
nuevas_columnas = [col for col in df_encoded.columns if any(var in col for var in variables_categoricas)]
print(f"\n📋 Nuevas columnas creadas ({len(nuevas_columnas)} total):")
for i, col in enumerate(nuevas_columnas, 1):
    print(f"   {i}. {col}")

In [ ]:
# Visualizar el resultado del encoding
print("\n" + "=" * 70)
print(" EJEMPLO DE DATOS TRANSFORMADOS")
print("=" * 70)

# Mostrar primeras filas con columnas relevantes
columnas_mostrar = ['Fecha', 'T_Ext_C', 'T_Int_C', 'Humedad_Pct', 'Altura_cm']
columnas_encoded = ['Responsable_Luis', 'Responsable_Juan', 'Fila_T0', 'Fila_T1', 'Planta_P1', 'Planta_P2']
columnas_mostrar += [col for col in columnas_encoded if col in df_encoded.columns]

print("\nPrimeras 10 filas del dataset transformado:")
df_encoded[columnas_mostrar].head(10)

In [ ]:
# Guardar datasets transformados
print("\n Guardando datasets...")

# Dataset limpio
df_clean.to_csv('Data_Mocollo1_Limpio.csv', index=False)
print("✅ Dataset limpio guardado: Data_Mocollo1_Limpio.csv")

# Dataset transformado con One Hot Encoding
df_encoded.to_csv('Data_Mocollo1_Transformado.csv', index=False)
print("✅ Dataset transformado guardado: Data_Mocollo1_Transformado.csv")

print("\n✨ Todos los datasets han sido guardados exitosamente")

---
## 5. Dashboard por Fecha <a id='dashboard'></a>

Creamos un dashboard interactivo para consultar información específica de una fecha.
En este ejemplo usamos la fecha: **22 de septiembre de 2026**

In [ ]:
# Configurar fecha de consulta
fecha_consulta = pd.to_datetime('2026-09-22')

print("=" * 80)
print(f" DASHBOARD - Fecha a Consultar: {fecha_consulta.strftime('%Y-%m-%d')}")
print("=" * 80)

# Filtrar datos de la fecha consultada
df_fecha = df_clean[df_clean['Fecha'] == fecha_consulta].copy()

if len(df_fecha) == 0:
    print(f"\n⚠️ No hay datos disponibles para la fecha {fecha_consulta.strftime('%d/%m/%Y')}")
    print("\n📅 Fechas disponibles:")
    for fecha in sorted(df_clean['Fecha'].unique()):
        print(f"  - {fecha.strftime('%d/%m/%Y')}")
else:
    print(f"\n✅ Datos encontrados: {len(df_fecha)} mediciones")
    
    # Calcular métricas resumen
    responsable = df_fecha['Responsable'].mode()[0] if 'Responsable' in df_fecha.columns else 'N/A'
    
    print("\n" + "=" * 80)
    print(" RESUMEN DE MÉTRICAS")
    print("=" * 80)
    print(f"\n👤 Responsable: {responsable}")
    print(f" Total de Plantas: {len(df_fecha)}")
    print(f"\n🌡️  TEMPERATURAS:")
    print(f"   • T° Externa Promedio: {df_fecha['T_Ext_C'].mean():.1f} °C")
    print(f"   • T° Interna Promedio: {df_fecha['T_Int_C'].mean():.1f} °C")
    print(f"   • T° Planta Promedio: {df_fecha['T_Planta_C'].mean():.1f} °C")
    print(f"\n💧 Humedad Promedio: {df_fecha['Humedad_Pct'].mean():.1f} %")
    print(f"🧪 pH Promedio: {df_fecha['pH'].mean():.1f}")
    print(f"📏 Altura Promedio: {df_fecha['Altura_cm'].mean():.1f} cm")

In [ ]:
# Dashboard visual completo
if len(df_fecha) > 0:
    # Crear figura con GridSpec para mejor control del layout
    fig = plt.figure(figsize=(16, 12))
    gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)
    
    # Título principal
    fig.suptitle(f' Dashboard - Agricultura Inteligente Macollo\nFecha: {fecha_consulta.strftime("%d de %B de %Y")}', 
                fontsize=18, fontweight='bold', y=0.98)
    
    # 1. Métricas principales (KPIs)
    ax1 = fig.add_subplot(gs[0, :])
    ax1.axis('off')
    
    # Crear tabla de métricas
    metricas = [
        ['👤 Responsable', df_fecha['Responsable'].mode()[0]],
        ['📏 Total Plantas', str(len(df_fecha))],
        ['️ T° Ext. Prom.', f"{df_fecha['T_Ext_C'].mean():.1f} °C"],
        ['🏠 T° Int. Prom.', f"{df_fecha['T_Int_C'].mean():.1f} °C"],
        ['💧 Humedad', f"{df_fecha['Humedad_Pct'].mean():.1f} %"],
        ['🌿 T° Planta Prom.', f"{df_fecha['T_Planta_C'].mean():.1f} °C"],
        ['🧪 pH Promedio', f"{df_fecha['pH'].mean():.1f}"],
        ['📏 Altura Prom.', f"{df_fecha['Altura_cm'].mean():.1f} cm"]
    ]
    
    tabla = ax1.table(cellText=metricas,
                     colLabels=['Métrica', 'Valor'],
                     cellLoc='center',
                     loc='center',
                     colColours=['#2E86AB', '#2E86AB'],
                     colWidths=[0.3, 0.2])
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(11)
    tabla.scale(1.2, 1.5)
    ax1.set_title('📊 Resumen de Métricas del Día', fontsize=14, fontweight='bold', pad=20)
    
    # 2. Distribución Espacial - Altura por Fila y Planta
    ax2 = fig.add_subplot(gs[1, :2])
    
    # Crear pivot table para el heatmap
    pivot_altura = df_fecha.pivot_table(values='Altura_cm', 
                                        index='Fila', 
                                        columns='Planta',
                                        aggfunc='mean')
    
    sns.heatmap(pivot_altura, annot=True, fmt='.0f', cmap='YlGnBu', 
               linewidths=0.5, ax=ax2, cbar_kws={'label': 'Altura (cm)'})
    ax2.set_title('🌿 Distribución Espacial del Invernadero - Altura (cm)', 
                 fontsize=12, fontweight='bold', pad=10)
    ax2.set_xlabel('Planta', fontweight='bold')
    ax2.set_ylabel('Fila / Treatment', fontweight='bold')
    
    # 3. Altura por Treatment (Gráfico de barras)
    ax3 = fig.add_subplot(gs[1, 2])
    
    altura_por_fila = df_fecha.groupby('Fila')['Altura_cm'].mean().reset_index()
    colors = plt.cm.Set3(np.linspace(0, 1, len(altura_por_fila)))
    
    sns.barplot(data=altura_por_fila, x='Fila', y='Altura_cm', 
               palette=colors, ax=ax3, edgecolor='black')
    ax3.set_title('📊 Altura Promedio por Treatment', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Fila', fontweight='bold')
    ax3.set_ylabel('Altura (cm)', fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Añadir valores en las barras
    for i, v in enumerate(altura_por_fila['Altura_cm']):
        ax3.text(i, v + 1, f'{v:.0f}', ha='center', fontweight='bold')
    
    # 4. Distribución de Alturas (Histograma + KDE)
    ax4 = fig.add_subplot(gs[2, 0])
    
    sns.histplot(data=df_fecha, x='Altura_cm', kde=True, 
                color='#4ECDC4', ax=ax4, bins=10)
    ax4.axvline(df_fecha['Altura_cm'].mean(), color='red', 
               linestyle='--', linewidth=2, label=f'Media: {df_fecha["Altura_cm"].mean():.1f}')
    ax4.set_title('📈 Distribución de Alturas', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Altura (cm)', fontweight='bold')
    ax4.set_ylabel('Frecuencia', fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Temperaturas (Boxplot)
    ax5 = fig.add_subplot(gs[2, 1])
    
    datos_temp = df_fecha[['T_Ext_C', 'T_Int_C', 'T_Planta_C']].melt(
        var_name='Tipo', value_name='Temperatura (°C)')
    datos_temp['Tipo'] = datos_temp['Tipo'].map({
        'T_Ext_C': 'Externa',
        'T_Int_C': 'Interna', 
        'T_Planta_C': 'Planta'
    })
    
    sns.boxplot(data=datos_temp, x='Tipo', y='Temperatura (°C)', 
               palette='coolwarm', ax=ax5)
    ax5.set_title('️ Distribución de Temperaturas', fontsize=12, fontweight='bold')
    ax5.set_xlabel('Tipo de Temperatura', fontweight='bold')
    ax5.set_ylabel('Temperatura (°C)', fontweight='bold')
    ax5.grid(True, alpha=0.3, axis='y')
    
    # 6. pH por Treatment (Gráfico de pastel)
    ax6 = fig.add_subplot(gs[2, 2])
    
    ph_por_fila = df_fecha.groupby('Fila')['pH'].mean()
    colors_pie = plt.cm.Pastel1(np.linspace(0, 1, len(ph_por_fila)))
    
    ax6.pie(ph_por_fila.values, labels=ph_por_fila.index, 
           autopct='%1.1f', colors=colors_pie, startangle=90)
    ax6.set_title(' pH Promedio por Treatment', fontsize=12, fontweight='bold')
    
    plt.show()
    
    # Tabla de detalle individual
    print("\n" + "=" * 80)
    print("📋 DETALLE DE MEDICIONES INDIVIDUALES DEL DÍA CONSULTADO")
    print("=" * 80)
    
    # Seleccionar y ordenar columnas relevantes
    columnas_detalle = ['Fila', 'Planta', 'ID_Planta', 'T_Planta_C', 'pH', 
                        'Altura_cm', 'T_Ext_C', 'T_Int_C', 'Humedad_Pct']
    
    df_detalle = df_fecha[columnas_detalle].copy()
    df_detalle = df_detalle.sort_values(['Fila', 'Planta'])
    
    # Mostrar tabla formateada
    print("\n")
    print(df_detalle.to_string(index=False, float_format="%.2f"))

In [ ]:
# Función reutilizable para consultar cualquier fecha
def consultar_fecha(fecha_str):
    """
    Función para consultar el dashboard de una fecha específica
    
    Parámetros:
    fecha_str: string con formato 'YYYY-MM-DD' o 'DD/MM/YYYY'
    """
    # Intentar convertir la fecha en ambos formatos
    try:
        fecha = pd.to_datetime(fecha_str)
    except:
        try:
            fecha = pd.to_datetime(fecha_str, format='%d/%m/%Y')
        except:
            print(f"❌ Formato de fecha no válido: {fecha_str}")
            return
    
    # Filtrar datos
    df_consulta = df_clean[df_clean['Fecha'] == fecha].copy()
    
    if len(df_consulta) == 0:
        print(f"⚠️ No hay datos para la fecha {fecha.strftime('%d/%m/%Y')}")
        return None
    
    print(f"\n✅ Datos encontrados para el {fecha.strftime('%d/%m/%Y')}: {len(df_consulta)} mediciones")
    
    return df_consulta

# Ejemplo de uso
print(" Para consultar otra fecha, use:")
print("   df_resultado = consultar_fecha('2026-09-22')")
print("   o")
print("   df_resultado = consultar_fecha('22/09/2026')")

print("\n📅 Fechas disponibles en el dataset:")
for i, fecha in enumerate(sorted(df_clean['Fecha'].unique()), 1):
    print(f"   {i}. {fecha.strftime('%d/%m/%Y')}")

---
## 6. Conclusiones y Recomendaciones

### 📌 Resumen del Análisis

1. **Calidad de Datos**: Se identificaron y limpiaron valores atípicos y formatos inconsistentes en las mediciones.

2. **Variables Categóricas**: Se transformaron exitosamente usando One Hot Encoding, generando columnas binarias para cada categoría.

3. **Patrones Temporales**: Se observó la evolución de las variables a lo largo del tiempo, permitiendo identificar tendencias.

4. **Distribución Espacial**: Las plantas muestran variaciones en altura según su ubicación (fila y treatment).

5. **Correlaciones**: Se identificaron relaciones importantes entre temperatura externa e interna.

### 🔧 Próximos Pasos

- Implementar modelos predictivos de crecimiento
- Crear alertas automáticas para valores fuera de rango
- Desarrollar dashboard interactivo en tiempo real
- Análisis de series temporales para predicción

### 📁 Archivos Generados

- `Data_Mocollo1_Limpio.csv`: Dataset limpio y procesado
- `Data_Mocollo1_Transformado.csv`: Dataset con One Hot Encoding aplicado

In [ ]:
# Celda final de resumen
print("=" * 80)
print("✨ ANÁLISIS COMPLETADO EXITOSAMENTE ✨")
print("=" * 80)
print("\n Resumen del proceso:")
print(f"   • Datos originales: {len(df)} registros")
print(f"   • Datos limpios: {len(df_clean)} registros")
print(f"   • Datos transformados: {len(df_encoded)} registros")
print(f"   • Variables categóricas codificadas: {len(variables_categoricas)}")
print(f"   • Visualizaciones generadas: 6+ gráficos")
print(f"   • Dashboard creado para: 22/09/2026")

print("\n🎯 Proyecto: Plataforma de Agricultura Inteligente Macollo")
print("📅 Fecha del análisis:", pd.Timestamp.now().strftime('%d/%m/%Y'))
print("\n✅ Todos los objetivos han sido completados")
print("=" * 80)